# Tokenizer adaptation experiments

T0/T1/T2 one config per run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, subprocess, sys
from pathlib import Path

def pip(args):
    print('pip', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip'] + args)

def install_tv():
    import torch
    torch_v = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
    tv_v = {'2.8': '0.23.0', '2.9': '0.24.0', '2.10': '0.25.0', '2.11': '0.26.0'}.get(torch_v)
    cuda = (torch.version.cuda or '').replace('.', '')
    if tv_v and cuda:
        pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', f'torchvision=={tv_v}+cu{cuda}', '--index-url', f'https://download.pytorch.org/whl/cu{cuda}'])
    elif tv_v:
        pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', f'torchvision=={tv_v}'])
    else:
        pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', 'torchvision'])

def tv_ok():
    try:
        from torchvision.ops import nms
        return True
    except Exception as e:
        print('torchvision broken:', repr(e))
        return False

def tr_ok():
    try:
        from transformers import ModernVBertConfig, ColModernVBertForRetrieval
        return True
    except Exception as e:
        print('transformers broken:', repr(e))
        return False

marker = Path('/content/drive/MyDrive/experiments_vbert/.deps_train_v7')
need_install = (not marker.exists()) or (not tv_ok()) or (not tr_ok())

if need_install:
    pip(['uninstall', '-y', 'torchao', 'torchvision', 'torchaudio'])
    pkgs = [
        'numpy==1.26.4', 'pandas==2.2.2', 'datasets==3.6.0', 'Pillow==11.3.0', 'tqdm==4.67.1', 'pyarrow==19.0.1', 'safetensors==0.5.3', 'fsspec[http]==2025.3.0', 'huggingface_hub>=1.5.0,<2.0', 'tokenizers>=0.22.0,<=0.23.0', 'requests==2.32.4', 'peft'
    ]
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall'] + pkgs)
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', 'git+https://github.com/huggingface/transformers'])
    install_tv()
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text('ok', encoding='utf-8')
    os.kill(os.getpid(), 9)

print('deps ok')


deps ok


In [ ]:
from pathlib import Path
import json, math, random, re, statistics, tarfile, time
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_from_disk
from PIL import Image
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, Idefics3ImageProcessor, ModernVBertConfig, ColModernVBertConfig, ColModernVBertForRetrieval, ColModernVBertProcessor

In [ ]:
SEED = 42
BASE_MODEL = 'ModernVBERT/colmodernvbert-merged'
EXPERIMENT_ID = 'T1-512TOK'
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
OUT = Path('/content/drive/MyDrive/experiments_vbert/training_runs') / EXPERIMENT_ID / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

VIDORE_TAR = Path('/content/drive/MyDrive/vidore_ru_result.tar')
VIDORE_ROOT = Path('/content/vidore_ru_result')
if not VIDORE_ROOT.exists():
    VIDORE_ROOT.mkdir(parents=True)
    with tarfile.open(VIDORE_TAR) as tar:
        tar.extractall(VIDORE_ROOT)

def found(paths):
    return sorted(paths, key=lambda p: len(p.parts))[0]

def hf_dir(name):
    hits = [p for p in VIDORE_ROOT.rglob(name) if p.is_dir() and (p / 'state.json').exists()]
    return found(hits or [p for p in VIDORE_ROOT.rglob(name) if p.is_dir()])

IMG_CACHE = {}

def fix_img(x):
    if not isinstance(x, str):
        return x
    p = Path(x)
    if p.exists():
        return str(p)

    parts = p.parts
    tries = [VIDORE_ROOT / p, Path('/content') / p, Path('/content/drive/MyDrive') / p]
    if parts and parts[0] == 'vidore_ru_work':
        tries.append(VIDORE_ROOT / Path(*parts[1:]))
    if 'ru_images' in parts:
        tries.append(VIDORE_ROOT / Path(*parts[parts.index('ru_images'):]))
    for q in tries:
        if q.exists():
            return str(q)

    if x not in IMG_CACHE:
        tail3 = '/'.join(parts[-3:])
        tail2 = '/'.join(parts[-2:])
        hits = [h for h in VIDORE_ROOT.rglob(p.name) if h.as_posix().endswith(tail3) or h.as_posix().endswith(tail2)]
        IMG_CACHE[x] = str(sorted(hits, key=lambda h: len(h.parts))[0]) if hits else x
    return IMG_CACHE[x]

def open_img(x):
    return x.convert('RGB') if isinstance(x, Image.Image) else Image.open(fix_img(x)).convert('RGB')

VIDORE_TRAIN_DIR = hf_dir('train_merged')
VIDORE_VAL_DIR = hf_dir('test_vidore_docvqa_test_subsampled')
VIDORE_TRAIN_JSONL = found(VIDORE_ROOT.rglob('train.jsonl'))
VIDORE_VAL_JSONL = found(VIDORE_ROOT.rglob('test_vidore_docvqa_test_subsampled.jsonl'))
OCR_TEXT_JSONL = VIDORE_TRAIN_JSONL
COCO_ID = 'romrawinjp/multilingual-coco'
COCO_TEST_FRACTION = 0.005
COCO_VAL_FRACTION = 0.002
ADD_TOKENS = 512  # use 1000 for T2, 0 for T0
LR = 2e-4
LORA_RANK = 16
MICRO_BATCH = 2
GRAD_ACCUM = 8
TRAIN_STEPS = 1000
EVAL_EVERY = 100
SMOKE_TEST = False
SMOKE_STEPS = None
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
def read_lines(path):
    if not path.exists():
        return []
    return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]

def text_corpus():
    texts = []
    if VIDORE_TRAIN_DIR.exists():
        try:
            ds = load_from_disk(str(VIDORE_TRAIN_DIR))
            rows = [ds[i] for i in range(min(len(ds), 256 if SMOKE_TEST else len(ds)))]
        except Exception:
            rows = read_lines(VIDORE_TRAIN_JSONL)
            rows = rows[:256] if SMOKE_TEST else rows
        for r in rows:
            texts.append(str(r.get('query_ru') or r.get('query') or ''))
    for r in read_lines(OCR_TEXT_JSONL):
        texts.append(str(r.get('query_ru') or r.get('query') or ''))
        texts.append(str(r.get('ocr_ru') or r.get('ocr_text') or ''))
    return [x.strip() for x in texts if x.strip()]

def cyr_words(texts):
    for t in texts:
        yield from re.findall(r'[А-Яа-яЁё]{2,}', t)

def fragmented_candidates(tok, texts, k):
    counts = Counter(w.lower() for w in cyr_words(texts))
    out = []
    for word, freq in counts.most_common():
        if freq < 2 or any(ch.isdigit() for ch in word):
            continue
        if len(tok.tokenize(word)) >= 4:
            out.append(word)
        if len(out) >= k:
            break
    return out

def tokenizer_metrics(tok, texts):
    ratios, qlens = [], []
    for t in texts:
        ws = list(cyr_words([t]))
        if ws:
            ratios += [len(tok.tokenize(w)) for w in ws]
        qlens.append(len(tok(t, add_special_tokens=False).input_ids))
    ratios = ratios or [0]
    return {
        'avg_tokens_per_cyrillic_word': float(np.mean(ratios)),
        'p50_tokens_per_cyrillic_word': float(np.percentile(ratios, 50)),
        'p90_tokens_per_cyrillic_word': float(np.percentile(ratios, 90)),
        'p95_tokens_per_cyrillic_word': float(np.percentile(ratios, 95)),
        'avg_query_sequence_length': float(np.mean(qlens or [0])),
        'avg_document_text_sequence_length': float(np.mean(qlens or [0])),
    }

In [ ]:
texts = text_corpus()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
before = tokenizer_metrics(tok, texts)
new_tokens = fragmented_candidates(tok, texts, ADD_TOKENS)
added = tok.add_tokens(new_tokens)

# mean(old subtoken embeddings) initialization hook; executed after model creation below
(OUT / 'tokenizer').mkdir(exist_ok=True)
tok.save_pretrained(OUT / 'tokenizer')
(OUT / 'added_tokens.json').write_text(json.dumps(new_tokens, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'tokenizer_metrics_before.json').write_text(json.dumps(before, ensure_ascii=False, indent=2), encoding='utf-8')
after = tokenizer_metrics(tok, texts)
(OUT / 'tokenizer_metrics_after.json').write_text(json.dumps(after, ensure_ascii=False, indent=2), encoding='utf-8')
print('added', added, 'tokens')

added 512 tokens


In [ ]:
def to_dev(batch, dev):
    return {k: v.to(dev) if hasattr(v, 'to') else v for k, v in batch.items()}

def remap_keys(sd):
    out = {}
    for k, v in sd.items():
        if k.startswith('model.vision_model.vision_model.'):
            k = k.replace('model.vision_model.vision_model.', 'vlm.vision_model.', 1)
        elif k.startswith('model.text_model.'):
            k = k.replace('model.text_model.', 'vlm.text_model.', 1)
        elif k.startswith('model.connector.'):
            k = k.replace('model.connector.', 'vlm.connector.', 1)
        elif k.startswith('custom_text_proj.'):
            k = k.replace('custom_text_proj.', 'embedding_proj_layer.', 1)
        out[k] = v
    return out

def make_processor(tok):
    image_processor = Idefics3ImageProcessor(
        do_convert_rgb=True, do_resize=True, size={'longest_edge': 2048},
        do_image_splitting=True, max_image_size={'longest_edge': 512},
        do_rescale=True, rescale_factor=1/255, do_normalize=True,
        image_mean=[0.5,0.5,0.5], image_std=[0.5,0.5,0.5], do_pad=True,
    )
    return ColModernVBertProcessor(image_processor=image_processor, tokenizer=tok, image_seq_len=64)

def load_token_model(tok, new_tokens):
    cfg = ColModernVBertConfig(vlm_config=ModernVBertConfig.from_pretrained(BASE_MODEL), embedding_dim=128)
    model = ColModernVBertForRetrieval(cfg)
    state = load_file(hf_hub_download(BASE_MODEL, 'model.safetensors'))
    missing, unexpected = model.load_state_dict(remap_keys(state), strict=False)
    missing = [x for x in missing if 'position_ids' not in x]
    if missing or unexpected:
        raise RuntimeError(f'bad load: missing={missing[:8]} unexpected={unexpected[:8]}')

    old_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for token in new_tokens:
            new_id = tok.convert_tokens_to_ids(token)
            old_ids = old_tok(token, add_special_tokens=False).input_ids
            if old_ids:
                emb[new_id] = emb[old_ids].mean(0)
    return model

def lora_model(model):
    for p in model.parameters():
        p.requires_grad = False
    model = get_peft_model(model, LoraConfig(r=LORA_RANK, lora_alpha=2*LORA_RANK, target_modules=['Wqkv','Wo','Wi'], bias='none'))
    emb = model.get_input_embeddings().weight
    emb.requires_grad = True
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.bfloat16 if dev == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
    return model.to(device=dev, dtype=dtype), dev

def vidore_rows(path, limit=None):
    try:
        ds = load_from_disk(str(path))
        raw = [ds[i] for i in range(min(len(ds), limit or len(ds)))]
    except Exception:
        jsonl = VIDORE_TRAIN_JSONL if 'train' in str(path) else VIDORE_VAL_JSONL
        if not jsonl.exists():
            raise
        raw = [json.loads(x) for x in jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]
        raw = raw[:limit] if limit else raw

    rows = []
    for r in raw:
        q = str(r.get('query_ru') or r.get('query') or '').strip()
        img = fix_img(r.get('image_ru_path') or r.get('image') or r.get('original_image_path'))
        if q and img:
            rows.append(dict(query=q, image=img))
    return rows

def coco_rows(part, limit=None):
    ds = load_dataset(COCO_ID, split='train').shuffle(seed=SEED)
    test_n = max(1, int(len(ds) * COCO_TEST_FRACTION))
    val_n = max(1, int(len(ds) * COCO_VAL_FRACTION))
    spans = {'test': (0, test_n), 'val': (test_n, test_n + val_n)}
    start, stop = spans[part]
    stop = min(stop, start + limit) if limit else stop
    out=[]
    for r in ds.select(range(start, stop)):
        ru=r.get('ru') or []
        if isinstance(ru,str): ru=[ru]
        ru=[str(x).strip() for x in ru if str(x).strip()]
        if ru: out.append(dict(query=ru[0],image=r['image']))
    return out

def split_emb(x,batch):
    mask=batch.get('attention_mask')
    return [x[i, mask[i].bool()] if mask is not None else x[i] for i in range(x.shape[0])]

def batch_loss(proc, model, dev, rows):
    queries=[r['query'] for r in rows]
    images=[open_img(r['image']) for r in rows]
    qb=to_dev(proc(text=queries, return_tensors='pt'),dev)
    db=to_dev(proc(text=['<image>']*len(images), images=images, return_tensors='pt'),dev)
    q=split_emb(model(**qb).embeddings,qb); d=split_emb(model(**db).embeddings,db)
    s=proc.score_retrieval(query_embeddings=q, passage_embeddings=d)
    return F.cross_entropy(s, torch.arange(len(rows),device=s.device))

def ndcg5(order,good):
    return 1/math.log2(order.index(good)+2) if good in order[:5] else 0.0

def eval_rows(proc,model,dev,rows):
    q=[]; d=[]; model.eval()
    for r in rows:
        qb=to_dev(proc(text=[r['query']], return_tensors='pt'),dev)
        img=open_img(r['image'])
        db=to_dev(proc(text=['<image>'], images=[img], return_tensors='pt'),dev)
        with torch.inference_mode(): q+=split_emb(model(**qb).embeddings,qb); d+=split_emb(model(**db).embeddings,db)
    scores=proc.score_retrieval(query_embeddings=q, passage_embeddings=d).detach().cpu()
    vals=[ndcg5(torch.argsort(scores[i],descending=True).tolist(), i) for i in range(len(rows))]
    model.train(); return float(np.mean(vals)) if vals else 0.0

proc = make_processor(tok)
model, dev = lora_model(load_token_model(tok, new_tokens))
train = vidore_rows(VIDORE_TRAIN_DIR, 32 if SMOKE_TEST else None)
vals = {'vidore_ru_val': vidore_rows(VIDORE_VAL_DIR, 16 if SMOKE_TEST else None), 'coco_ru_val': coco_rows('val', 16 if SMOKE_TEST else None)}
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = SMOKE_STEPS if SMOKE_TEST else TRAIN_STEPS
history=[]; best=-1; best_payload=None
for step in tqdm(range(1,steps+1),desc='token train'):
    rows=random.choices(train,k=MICRO_BATCH)
    loss=batch_loss(proc,model,dev,rows)/GRAD_ACCUM; loss.backward()
    if step % GRAD_ACCUM == 0 or step == steps: opt.step(); opt.zero_grad()
    rec={'step':step,'loss':float(loss.detach().cpu())*GRAD_ACCUM}
    if step % (1 if SMOKE_TEST else EVAL_EVERY)==0 or step==steps:
        scores={k:eval_rows(proc,model,dev,v) for k,v in vals.items()}; mean=float(np.mean(list(scores.values()))); rec.update(scores); rec['mean_ndcg5']=mean
        if mean>best:
            best=mean; best_payload={'step':step,'mean_ndcg5':mean,**scores}; model.save_pretrained(OUT/'adapter')
    history.append(rec)

training_config={k:v for k,v in globals().items() if k.isupper() and isinstance(v,(str,int,float,bool))}
(OUT/'training_config.json').write_text(json.dumps(training_config,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'val_metrics.json').write_text(json.dumps(best_payload or {},ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'best_checkpoint_meta.json').write_text(json.dumps(best_payload or {},ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
print('saved tokenizer training run to', OUT)

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

token train:   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(
